# Local execution

A bounded subprocess stops a trusted worker after a deadline. `ShellToolMiddleware` instead provides an agent with a persistent shell session through `HostExecutionPolicy`. Both execute with the host user's permissions.

[Host execution](https://docs.langchain.com/oss/python/langchain/middleware/built-in#shell-tool)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.config import settings
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.faults import scripted_model
from lab.sandbox import clean_environment, local_shell, run_worker, workspace
from langchain.agents import create_agent
from langchain.agents.middleware import HostExecutionPolicy, ShellToolMiddleware
from model_config import get_model, run_config
from lab.faults import ScriptedModel, tool_request
from langchain_core.messages import AIMessage
from langchain.agents.middleware.shell_tool import RedactionRule


## Bounded worker processes

A separate Python process runs a calculation, an endless sleep or an endless CPU loop. The calculation returns `42`; a deadline terminates each nonterminating worker. This fixture launcher accepts only its three known worker names.

In [ ]:
print("Local timeout:", settings()["sandbox"]["command_timeout"], "seconds")
local_hello = run_worker("hello", mode="local")
print(local_hello)
local_wait_forever = run_worker("wait_forever", mode="local")
print(local_wait_forever)
local_spin_forever = run_worker("spin_forever", mode="local")
print(local_spin_forever)
assert local_hello["status"] == "completed" and "42" in local_hello["stdout"]
assert local_wait_forever["status"] == local_spin_forever["status"] == "timed_out"

## An agent-operated host shell

The model requests `python -c "print(6 * 7)"`. The shell executes it, returns `42` as a tool result, and the model answers. The working directory selects where commands begin; it does not restrict access to the rest of the host filesystem.

In [ ]:
# Execute
host_shell = ShellToolMiddleware(
    workspace_root=workspace(), shell_command=local_shell(),
    env=clean_environment(),
    execution_policy=HostExecutionPolicy(command_timeout=settings()["sandbox"]["command_timeout"]),
)
agent = create_agent(
    model=scripted_model("host_shell"),
    middleware=[host_shell, TraceMiddleware("09_host_shell", pretty=True)],
)
result = agent.invoke({"messages": [("user", prompt("shell_hello"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)

## Execution boundary

Host execution uses the account's filesystem and network permissions. A timeout bounds elapsed time but does not impose CPU, memory or process-count limits. Terminating one subprocess is not equivalent to containing an arbitrary descendant process tree. A model-call limit cannot interrupt a tool that has already blocked.

### Persistent shell state, lifecycle commands and redaction

Commands within one agent invocation share a shell session, including environment variables and working directory. Startup commands initialize the session, shutdown commands finalize it, and redaction rules replace matching output before it reaches the model. Redaction is post-execution; it does not restrict what a host process can access.

In [ ]:
# Execute
session_root = workspace()
session_shell = ShellToolMiddleware(workspace_root=session_root, shell_command=local_shell(), env=clean_environment(),
    startup_commands=["export WORKSHOP_VALUE=42"], shutdown_commands=["printf done > session-ended.txt"],
    redaction_rules=[RedactionRule(pii_type="secret", detector=r"secret-demo")],
    execution_policy=HostExecutionPolicy(command_timeout=settings()["sandbox"]["command_timeout"]),
)
session_agent = create_agent(model=ScriptedModel(responses=[
    tool_request("shell", command="export WORKSHOP_CARRY=retained"),
    tool_request("shell", command='echo "$WORKSHOP_VALUE $WORKSHOP_CARRY secret-demo"'),
    AIMessage(content="The shell retained both variables and redacted its output."),
]), middleware=[session_shell])
session_result = session_agent.invoke({"messages": [("user", "Demonstrate shell state.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(session_result["messages"], pretty=True)
assert any("42 retained" in m.content and "[REDACTED_SECRET]" in m.content for m in session_result["messages"] if m.type == "tool")
assert (session_root / "session-ended.txt").exists()